In [1]:
import pandas as pd
import requests
import json
import time
from pathlib import Path
from datetime import datetime, timezone

In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELING_DIR = PROJECT_ROOT / "data" / "modeling"

print("Project root:", PROJECT_ROOT)
print("Pandas version:", pd.__version__)

Project root: /Users/xinyishen/Desktop/Brown/DATA 1030/DATA1030-fall2026/DATA1030-Midterm-Xinfinity
Pandas version: 3.0.3


In [3]:
BASE_URL = "https://api.nhtsa.gov/complaints/complaintsByVehicle"

In [4]:
#test API connection
params = {
    "make": "Toyota",
    "model": "Camry",
    "modelYear": 2020
}

response = requests.get(
    BASE_URL,
    params=params,
    timeout=30
)

response.raise_for_status()

data = response.json()

print("Status Code:", response.status_code)
print("Response Keys:", list(data.keys()))
print("Number of complaints:", len(data.get("results", [])))

Status Code: 200
Response Keys: ['count', 'message', 'results']
Number of complaints: 275


In [5]:
complaints = data["results"]

df_raw = pd.DataFrame(complaints)

print("Number of rows:", df_raw.shape[0])
print("Number of columns:", df_raw.shape[1])

print("\nAvailable columns:")
print(df_raw.columns.tolist())

display(df_raw.head(3))

Number of rows: 275
Number of columns: 12

Available columns:
['odiNumber', 'manufacturer', 'crash', 'fire', 'numberOfInjuries', 'numberOfDeaths', 'dateOfIncident', 'dateComplaintFiled', 'vin', 'components', 'summary', 'products']


,odiNumber,manufacturer,crash,fire,numberOfInjuries,numberOfDeaths,dateOfIncident,dateComplaintFiled,vin,components,summary,products
0,11770517,Toyota Motor Corporation,False,False,0,0,10/05/2026,10/12/2026,4T1C11AK2LU,"ELECTRICAL SYSTEM,AIR BAGS,UNKNOWN OR OTHER","The ""airbag malfunction"" light keeps coming on...","[{'type': 'Vehicle', 'productYear': '2020', 'p..."
1,11766564,Toyota Motor Corporation,False,False,0,0,09/05/2026,09/23/2026,4T1C11AK1LU,FUEL/PROPULSION SYSTEM,"While fixing another problem, I pulled the cod...","[{'type': 'Vehicle', 'productYear': '2020', 'p..."
2,11765004,Toyota Motor Corporation,False,False,0,0,06/01/2021,09/17/2026,4T1K61AK7LU,"ELECTRICAL SYSTEM,UNKNOWN OR OTHER,FUEL/PROPUL...",My car sputters and dies while driving. It loo...,"[{'type': 'Vehicle', 'productYear': '2020', 'p..."


In [6]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 275 entries, 0 to 274
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   odiNumber           275 non-null    int64 
 1   manufacturer        275 non-null    str   
 2   crash               275 non-null    bool  
 3   fire                275 non-null    bool  
 4   numberOfInjuries    275 non-null    int64 
 5   numberOfDeaths      275 non-null    int64 
 6   dateOfIncident      275 non-null    str   
 7   dateComplaintFiled  275 non-null    str   
 8   vin                 275 non-null    str   
 9   components          275 non-null    str   
 10  summary             275 non-null    str   
 11  products            275 non-null    object
dtypes: bool(2), int64(3), object(1), str(6)
memory usage: 22.2+ KB


In [7]:
# Missing value summary
missing_summary = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_pct": (
        df_raw.isna().mean() * 100
    ).round(2)
}).sort_values("missing_pct", ascending=False)

display(missing_summary)

,missing_count,missing_pct
odiNumber,0,0.0
manufacturer,0,0.0
crash,0,0.0
fire,0,0.0
numberOfInjuries,0,0.0
numberOfDeaths,0,0.0
dateOfIncident,0,0.0
dateComplaintFiled,0,0.0
vin,0,0.0
components,0,0.0


In [8]:
print("API reported count:", data.get("count"))
print("Records retrieved:", len(df_raw))

assert data.get("count") == len(df_raw), (
    "API count does not match retrieved records"
)

print("PASS: API count matches retrieved records")

API reported count: 275
Records retrieved: 275
PASS: API count matches retrieved records


In [9]:
for col in df_raw.columns:
    print(f"\n--- {col} ---")
    print("Unique values:", df_raw[col].astype(str).nunique())
    print(df_raw[col].astype(str).value_counts().head(15))


--- odiNumber ---
Unique values: 275
odiNumber
11770517    1
11766564    1
11765004    1
11764646    1
11762945    1
11760803    1
11760393    1
11756265    1
11754649    1
11754116    1
11753736    1
11752925    1
11747189    1
11746558    1
11745192    1
Name: count, dtype: int64

--- manufacturer ---
Unique values: 1
manufacturer
Toyota Motor Corporation    275
Name: count, dtype: int64

--- crash ---
Unique values: 2
crash
False    261
True      14
Name: count, dtype: int64

--- fire ---
Unique values: 2
fire
False    273
True       2
Name: count, dtype: int64

--- numberOfInjuries ---
Unique values: 4
numberOfInjuries
0    263
2      6
1      4
3      2
Name: count, dtype: int64

--- numberOfDeaths ---
Unique values: 1
numberOfDeaths
0    275
Name: count, dtype: int64

--- dateOfIncident ---
Unique values: 254
dateOfIncident
12/20/2023    5
08/01/2020    3
09/02/2020    3
03/11/2026    2
12/31/1969    2
01/05/2026    2
12/15/2025    2
12/04/2025    2
11/11/2025    2
01/13/2025   

## Choose variables

In [10]:
vehicles = {
    "Toyota": ["Camry"],
    "Honda": ["Civic"],
    "Ford": ["Escape", "F-150"],
    "Hyundai": ["Elantra"],
    "Nissan": ["Altima"],
    "BMW": ["3 Series"],
    "Mercedes-Benz": ["C-Class"],
    "Audi": ["Q5"],
    "Volkswagen": ["Tiguan"]
}
years = [2018, 2019, 2020]

print("Planned requests:", sum(len(models) for models in vehicles.values()) * len(years))

Planned requests: 30


## Get complaints from the API

In [11]:
# record API retrieval date
retrieval_date = datetime.now(timezone.utc)

print("API retrieval date (UTC):", retrieval_date)

API retrieval date (UTC): 2026-10-14 21:49:03.049725+00:00


In [12]:
all_records = []
request_log = []

for make, models in vehicles.items():
    for model in models:
        for year in years:
            params = {"make": make, "model": model, "modelYear": year}
            try:
                response = requests.get(BASE_URL, params=params, timeout=60)
                response.raise_for_status()
                result = response.json()
                records = result["results"]
                if result["count"] != len(records):
                    raise ValueError("API count does not match")

                for record in records:
                    row = record.copy()
                    row["query_make"] = make
                    row["query_model"] = model
                    row["query_model_year"] = year
                    all_records.append(row)

                status = "success"
                print(make, model, year, len(records))
            except (requests.RequestException, ValueError, KeyError, TypeError) as e:
                status = "failed"
                records = []
                print("FAILED:", make, model, year, e)

            request_log.append({"make": make, "model": model,
                                "year": year, "count": len(records),
                                "status": status})
            time.sleep(0.5)

df_raw = pd.DataFrame(all_records)
df_request_log = pd.DataFrame(request_log)
display(df_request_log)

Toyota Camry 2018 748
Toyota Camry 2019 397
Toyota Camry 2020 275
Honda Civic 2018 596
Honda Civic 2019 346
Honda Civic 2020 228
Ford Escape 2018 1811
Ford Escape 2019 595
Ford Escape 2020 1537
Ford F-150 2018 84
Ford F-150 2019 58
Ford F-150 2020 50
Hyundai Elantra 2018 283
Hyundai Elantra 2019 251
Hyundai Elantra 2020 253
Nissan Altima 2018 226
Nissan Altima 2019 232
Nissan Altima 2020 179
BMW 3 Series 2018 59
BMW 3 Series 2019 41
BMW 3 Series 2020 53
Mercedes-Benz C-Class 2018 130
Mercedes-Benz C-Class 2019 79
Mercedes-Benz C-Class 2020 46
Audi Q5 2018 207
Audi Q5 2019 52
Audi Q5 2020 25
Volkswagen Tiguan 2018 286
Volkswagen Tiguan 2019 193
Volkswagen Tiguan 2020 141


,make,model,year,count,status
0,Toyota,Camry,2018,748,success
1,Toyota,Camry,2019,397,success
2,Toyota,Camry,2020,275,success
3,Honda,Civic,2018,596,success
4,Honda,Civic,2019,346,success
5,Honda,Civic,2020,228,success
6,Ford,Escape,2018,1811,success
7,Ford,Escape,2019,595,success
8,Ford,Escape,2020,1537,success
9,Ford,F-150,2018,84,success


In [13]:
# Check that all API requests were successful
assert len(df_request_log) == 30
assert (df_request_log["status"] == "success").all()

print("All 30 API requests completed successfully!")

All 30 API requests completed successfully!


In [14]:
print("Requests:", len(df_request_log))
print("Failed requests:", (df_request_log["status"] == "failed").sum())
print("Zero-result requests:", ((df_request_log["status"] == "success") & (df_request_log["count"] == 0)).sum())
display(df_request_log[df_request_log["status"] != "success"])
display(df_request_log[(df_request_log["status"] == "success") & (df_request_log["count"] == 0)])

print("Rows:", len(df_raw))
if not df_raw.empty:
    print("Unique complaint IDs:", df_raw["odiNumber"].nunique())
    print("Repeated IDs across queries:", df_raw["odiNumber"].duplicated().sum())

Requests: 30
Failed requests: 0
Zero-result requests: 0


,make,model,year,count,status


,make,model,year,count,status


Rows: 9461
Unique complaint IDs: 9461
Repeated IDs across queries: 0


In [15]:
display(df_raw.drop(columns=["vin", "summary", "products"], errors="ignore").head())
print(df_raw.shape)
df_raw.info()

,odiNumber,manufacturer,crash,fire,numberOfInjuries,numberOfDeaths,dateOfIncident,dateComplaintFiled,components,query_make,query_model,query_model_year
0,11770133,Toyota Motor Corporation,False,False,0,0,09/29/2026,10/09/2026,POWER TRAIN,Toyota,Camry,2018
1,11769969,Toyota Motor Corporation,False,False,0,0,10/06/2026,10/08/2026,ENGINE,Toyota,Camry,2018
2,11769124,Toyota Motor Corporation,False,False,0,0,10/03/2026,10/05/2026,POWER TRAIN,Toyota,Camry,2018
3,11768868,Toyota Motor Corporation,False,False,0,0,10/02/2026,10/04/2026,FUEL/PROPULSION SYSTEM,Toyota,Camry,2018
4,11768335,Toyota Motor Corporation,False,False,0,0,09/29/2026,10/01/2026,POWER TRAIN,Toyota,Camry,2018


(9461, 15)
<class 'pandas.DataFrame'>
RangeIndex: 9461 entries, 0 to 9460
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   odiNumber           9461 non-null   int64 
 1   manufacturer        9461 non-null   str   
 2   crash               9461 non-null   bool  
 3   fire                9461 non-null   bool  
 4   numberOfInjuries    9461 non-null   int64 
 5   numberOfDeaths      9461 non-null   int64 
 6   dateOfIncident      9461 non-null   str   
 7   dateComplaintFiled  9461 non-null   str   
 8   vin                 9438 non-null   str   
 9   components          9461 non-null   str   
 10  summary             9461 non-null   str   
 11  products            9461 non-null   object
 12  query_make          9461 non-null   str   
 13  query_model         9461 non-null   str   
 14  query_model_year    9461 non-null   int64 
dtypes: bool(2), int64(4), object(1), str(8)
memory usage: 979.5+ KB


## Check for missing values and distributions

In [16]:
print("Pandas missing values:")
display(df_raw.isna().sum())

print("Blank strings:")
text_cols = ["manufacturer", "vin", "components", "summary"]
for col in text_cols:
    if col in df_raw:
        print(col, df_raw[col].astype("string").str.strip().eq("").sum())

if "products" in df_raw:
    print("Empty product lists:", df_raw["products"].map(lambda x: isinstance(x, list) and len(x) == 0).sum())

Pandas missing values:


odiNumber              0
manufacturer           0
crash                  0
fire                   0
numberOfInjuries       0
numberOfDeaths         0
dateOfIncident         0
dateComplaintFiled     0
vin                   23
components             0
summary                0
products               0
query_make             0
query_model            0
query_model_year       0
dtype: int64

Blank strings:
manufacturer 0
vin 176
components 0
summary 0
Empty product lists: 0


In [17]:
for col in ["manufacturer", "crash", "fire", "numberOfInjuries", "numberOfDeaths", "components"]:
    print("\n", col)
    display(df_raw[col].value_counts(dropna=False).head(10))


 manufacturer


manufacturer
Ford Motor Company                    4135
Toyota Motor Corporation              1420
Honda (American Honda Motor Co.)      1170
Volkswagen Group of America, Inc.      904
Hyundai Motor America                  787
Nissan North America, Inc.             637
Mercedes-Benz USA, LLC                 254
BMW of North America, LLC              153
Continental Tire the Americas, LLC       1
Name: count, dtype: int64


 crash


crash
False    9092
True      369
Name: count, dtype: int64


 fire


...

numberOfInjuries
0     9218
1      182
2       49
4        5
3        4
10       1
6        1
5        1
Name: count, dtype: int64


 numberOfDeaths


numberOfDeaths
0    9456
1       4
2       1
Name: count, dtype: int64


 components


components
ENGINE                       1719
POWER TRAIN                   820
UNKNOWN OR OTHER              761
STRUCTURE                     641
STEERING                      424
ENGINE AND ENGINE COOLING     372
ELECTRICAL SYSTEM             351
FUEL SYSTEM, GASOLINE         297
FUEL/PROPULSION SYSTEM        270
SERVICE BRAKES                208
Name: count, dtype: int64

## Check the vehicle info in `products`

In [18]:
matched = []
for _, row in df_raw.iterrows():
    found = False
    if isinstance(row["products"], list):
        for p in row["products"]:
            if (isinstance(p, dict)
                and p.get("type") == "Vehicle"
                and str(p.get("productMake", "")).upper() == row["query_make"].upper()
                and str(p.get("productModel", "")).upper() == row["query_model"].upper()
                and str(p.get("productYear", "")) == str(row["query_model_year"])):
                found = True
    matched.append(found)

df_raw["vehicle_match"] = matched
print(df_raw["vehicle_match"].value_counts(dropna=False))
display(df_raw.loc[~df_raw["vehicle_match"],
                   ["odiNumber", "query_make", "query_model", "query_model_year"]].head(10))

vehicle_match
True     9053
False     408
Name: count, dtype: int64


,odiNumber,query_make,query_model,query_model_year
8149,11756464,BMW,3 Series,2018
8150,11678001,BMW,3 Series,2018
8151,11670096,BMW,3 Series,2018
8152,11667410,BMW,3 Series,2018
8153,11660219,BMW,3 Series,2018
8154,11655822,BMW,3 Series,2018
8155,11648186,BMW,3 Series,2018
8156,11646108,BMW,3 Series,2018
8157,11641715,BMW,3 Series,2018
8158,11637952,BMW,3 Series,2018


In [19]:
df_raw.groupby(
    ["query_make", "query_model"]
)["vehicle_match"].value_counts().unstack(fill_value=0)

vehicle_match              True   False
query_make    query_model              
Audi          Q5             284      0
BMW           3 Series         0    153
Ford          Escape        3943      0
              F-150          192      0
Honda         Civic         1170      0
Hyundai       Elantra        787      0
Mercedes-Benz C-Class          0    255
Nissan        Altima         637      0
Toyota        Camry         1420      0
Volkswagen    Tiguan         620      0

In [20]:
for model in ["3 Series", "C-Class"]:
    subset = df_raw[
        df_raw["query_model"] == model
    ]

    print(f"\n--- {model} ---")

    print("Example products:")
    display(subset["products"].head(3).tolist())


--- 3 Series ---
Example products:


[[{'type': 'Vehicle',
   'productYear': '2018',
   'productMake': 'BMW',
   'productModel': '330E',
   'manufacturer': 'BMW of North America, LLC'}],
 [{'type': 'Vehicle',
   'productYear': '2018',
   'productMake': 'BMW',
   'productModel': '330I',
   'manufacturer': 'BMW of North America, LLC'}],
 [{'type': 'Vehicle',
   'productYear': '2018',
   'productMake': 'BMW',
   'productModel': '328D',
   'manufacturer': 'BMW of North America, LLC'}]]


--- C-Class ---
Example products:


[[{'type': 'Vehicle',
   'productYear': '2018',
   'productMake': 'MERCEDES-BENZ',
   'productModel': 'C 300',
   'manufacturer': 'Mercedes-Benz USA, LLC'}],
 [{'type': 'Vehicle',
   'productYear': '2018',
   'productMake': 'MERCEDES BENZ',
   'productModel': 'C 300 4MATIC',
   'manufacturer': 'Mercedes-Benz USA, LLC'}],
 [{'type': 'Vehicle',
   'productYear': '2018',
   'productMake': 'MERCEDES-BENZ',
   'productModel': 'C 300',
   'manufacturer': 'Mercedes-Benz USA, LLC'}]]

## Identify falsely categorized

In [21]:
# Get vehicle model from products
def get_vehicle_model(products):
    for p in products:
        if p.get("type") == "Vehicle":
            return p.get("productModel")
    return None

df_raw["product_model"] = df_raw["products"].apply(get_vehicle_model)

In [22]:
display(
    df_raw.groupby(["query_make", "product_model"])
    .size()
    .reset_index(name="count")
)

,query_make,product_model,count
0,Audi,Q5,284
1,BMW,320I,16
2,BMW,328D,7
3,BMW,330E,6
4,BMW,330I,121
5,BMW,340I,3
6,Ford,ESCAPE,3943
7,Ford,F-150,192
8,Honda,CIVIC,1170
9,Hyundai,ELANTRA,787


In [23]:
model_mapping = {
    "320I": "3 SERIES",
    "328D": "3 SERIES",
    "330E": "3 SERIES",
    "330I": "3 SERIES",
    "340I": "3 SERIES",

    "AMG C43": "C-CLASS",
    "AMG C63": "C-CLASS",
    "AMG C63 S": "C-CLASS",
    "AMG C63S": "C-CLASS",
    "C 250 D": "C-CLASS",
    "C 300": "C-CLASS",
    "C 300 4MATIC": "C-CLASS",
    "C 350 E": "C-CLASS",
    "C300": "C-CLASS",
    "C350": "C-CLASS",
    "C205": "C-CLASS"
}

In [24]:
df_clean = df_raw.copy()

df_clean["product_model"] = (
    df_clean["product_model"].str.upper().str.strip()
)

df_clean["make"] = (
    df_clean["query_make"].str.upper().str.strip()
)

df_clean["model"] = (
    df_clean["product_model"]
    .replace(model_mapping)
)

df_clean["model_year"] = df_clean["query_model_year"]

In [25]:
display(
    df_clean.groupby(["make", "model"])
    .size()
    .reset_index(name="count")
)

,make,model,count
0,AUDI,Q5,284
1,BMW,3 SERIES,153
2,FORD,ESCAPE,3943
3,FORD,F-150,192
4,HONDA,CIVIC,1170
5,HYUNDAI,ELANTRA,787
6,MERCEDES-BENZ,C-CLASS,255
7,NISSAN,ALTIMA,637
8,TOYOTA,CAMRY,1420
9,VOLKSWAGEN,TIGUAN,620


In [26]:
df_raw.loc[
    df_raw["product_model"] == "PROCONTACT GX SSR MOE",
    ["odiNumber", "query_make", "query_model", "products"]
]

,odiNumber,query_make,query_model,products


In [27]:
# df_raw.loc[8386, "products"]

In [28]:
df_raw.loc[
    df_raw["product_model"] == "C205",
    ["odiNumber", "query_model_year", "products"]
]

,odiNumber,query_model_year,products
8453,11680791,2019,"[{'type': 'Vehicle', 'productYear': '2019', 'p..."


In [29]:
df_raw.loc[8435, "products"]

[{'type': 'Vehicle',
  'productYear': '2019',
  'productMake': 'MERCEDES-BENZ',
  'productModel': 'C 300',
  'manufacturer': 'Mercedes-Benz USA, LLC'}]

In [30]:
df_clean.groupby(["make", "model"]).size()

make           model   
AUDI           Q5           284
BMW            3 SERIES     153
FORD           ESCAPE      3943
               F-150        192
HONDA          CIVIC       1170
HYUNDAI        ELANTRA      787
MERCEDES-BENZ  C-CLASS      255
NISSAN         ALTIMA       637
TOYOTA         CAMRY       1420
VOLKSWAGEN     TIGUAN       620
dtype: int64

# Data Cleaning

## Convert dates & Find delays

In [31]:
df_clean["filed_date"] = pd.to_datetime(
    df_clean["dateComplaintFiled"],
    format="%m/%d/%Y",
    errors="coerce"
)

df_clean["incident_date"] = pd.to_datetime(
    df_clean["dateOfIncident"],
    format="%m/%d/%Y",
    errors="coerce"
)

In [32]:
# record filling date range
earliest_date = df_clean["filed_date"].min()
latest_date = df_clean["filed_date"].max()

print("Earliest filing date:", earliest_date)
print("Latest filing date:", latest_date)

Earliest filing date: 2017-08-04 00:00:00
Latest filing date: 2026-10-12 00:00:00


In [33]:
print("Missing filed dates:", df_clean["filed_date"].isna().sum())
print("Missing incident dates:", df_clean["incident_date"].isna().sum())

print("Implausible incident dates:",
      (df_clean["incident_date"].dt.year <
       df_clean["model_year"] - 2).sum())

print("Incident after filing:",
      (df_clean["incident_date"] > df_clean["filed_date"]).sum())

Missing filed dates: 0
Missing incident dates: 0
Implausible incident dates: 27
Incident after filing: 0


In [34]:
df_clean["reporting_delay"] = (
    df_clean["filed_date"] - df_clean["incident_date"]
).dt.days

df_clean["reporting_delay"].describe()

count     9461.000000
mean       137.400486
std       1005.970404
min          0.000000
25%          2.000000
50%         13.000000
75%         61.000000
max      37015.000000
Name: reporting_delay, dtype: float64

In [35]:
df_clean.loc[
    df_clean["incident_date"].dt.year < df_clean["model_year"] - 2,
    ["odiNumber", "make", "model", "model_year",
     "incident_date", "filed_date", "reporting_delay"]
]

,odiNumber,make,model,model_year,incident_date,filed_date,reporting_delay
89,11702696,TOYOTA,CAMRY,2018,1969-12-31,2025-12-03,20426
127,11661096,TOYOTA,CAMRY,2018,2015-04-02,2025-05-15,3696
460,11372676,TOYOTA,CAMRY,2018,2013-01-01,2020-11-02,2862
1032,11387502,TOYOTA,CAMRY,2019,1919-09-09,2021-01-11,37015
1173,11718234,TOYOTA,CAMRY,2020,1969-12-31,2026-02-16,20501
1198,11690118,TOYOTA,CAMRY,2020,1969-12-31,2025-09-27,20359
2380,11736426,HONDA,CIVIC,2020,1969-12-31,2026-05-07,20581
2709,11746148,FORD,ESCAPE,2018,1969-12-31,2026-06-24,20629
2760,11737407,FORD,ESCAPE,2018,1969-12-31,2026-05-12,20586
2781,11733178,FORD,ESCAPE,2018,1969-12-31,2026-04-23,20567


In [36]:
df_clean["invalid_incident_date"] = (
    df_clean["incident_date"].dt.year
    < df_clean["model_year"] - 2
)

print("Flagged incident dates:",
      df_clean["invalid_incident_date"].sum())

Flagged incident dates: 27


### Date Validation Summary

Incident dates were checked against vehicle model years and complaint filing dates.

27 incident dates were flagged as implausible because they occurred more than two years before the vehicle model year.

* 0 incident dates occurred after their complaint filing dates.

* 0 future incident dates were found.

* 0 negative reporting delays were found.

* 23 reporting delays exceeded five years after excluding flagged dates.

Original incident dates were preserved. Reporting delays for flagged records were treated as missing, while long but potentially valid reporting delays were retained for further EDA.

In [37]:
df_clean["reporting_delay_clean"] = (
    df_clean["reporting_delay"].where(
        ~df_clean["invalid_incident_date"]
    )
)

df_clean["reporting_delay_clean"].describe()

count    9434.000000
mean       89.878100
std       229.309075
min         0.000000
25%         2.000000
50%        12.000000
75%        60.750000
max      2922.000000
Name: reporting_delay_clean, dtype: float64

In [38]:
# Additional Date validation
today = pd.Timestamp.now()

print("Incident dates after filing:", (df_clean["incident_date"] > df_clean["filed_date"]).sum())

print("Future incident dates:", (df_clean["incident_date"] > today).sum())

print("Negative reporting delays:", (df_clean["reporting_delay"] < 0).sum())

print("Reporting delays over 5 years:", (df_clean["reporting_delay_clean"] > 365 * 5).sum())

Incident dates after filing: 0
Future incident dates: 0
Negative reporting delays: 0
Reporting delays over 5 years: 23


In [39]:
long_delay = df_clean[
    df_clean["reporting_delay_clean"] > 365 * 5
]

display(
    long_delay[
        ["odiNumber", "make", "model", "model_year",
         "incident_date", "filed_date",
         "reporting_delay_clean"]
    ].sort_values("reporting_delay_clean", ascending=False)
)

,odiNumber,make,model,model_year,incident_date,filed_date,reporting_delay_clean
1593,11602285,HONDA,CIVIC,2018,2016-07-15,2024-07-15,2922.0
8849,11759418,VOLKSWAGEN,TIGUAN,2018,2019-05-01,2026-08-25,2673.0
6646,11757482,FORD,F-150,2019,2019-05-14,2026-08-17,2652.0
43,11736083,TOYOTA,CAMRY,2018,2019-05-06,2026-05-06,2557.0
1477,11649712,HONDA,CIVIC,2018,2018-03-21,2025-03-21,2557.0
6641,11760295,FORD,F-150,2019,2020-03-30,2026-08-28,2342.0
1669,11584103,HONDA,CIVIC,2018,2017-12-15,2024-04-19,2317.0
2919,11698729,FORD,ESCAPE,2018,2019-08-01,2025-11-11,2294.0
6535,11769433,FORD,F-150,2018,2020-07-01,2026-10-06,2288.0
4681,11638443,FORD,ESCAPE,2019,2019-04-13,2025-01-24,2113.0


In [40]:
print("Total rows:", len(df_clean))
print("Unique ODI numbers:", df_clean["odiNumber"].nunique())
print("Duplicate ODI numbers:",
      df_clean["odiNumber"].duplicated().sum())

Total rows: 9461
Unique ODI numbers: 9461
Duplicate ODI numbers: 0


In [41]:
duplicates = df_clean[
    df_clean.duplicated("odiNumber", keep=False)
]

display(duplicates[
    ["odiNumber", "make", "model",
     "model_year", "components"]
].head(20))

,odiNumber,make,model,model_year,components


## Missing value handling

In [42]:
df_clean["vin"] = df_clean["vin"].replace(
    r"^\s*$", pd.NA, regex=True
)

print("Missing VIN:", df_clean["vin"].isna().sum())

Missing VIN: 199


In [43]:
df_clean.isna().sum().sort_values(ascending=False)

vin                      199
reporting_delay_clean     27
query_model                0
invalid_incident_date      0
reporting_delay            0
incident_date              0
filed_date                 0
model_year                 0
model                      0
make                       0
product_model              0
vehicle_match              0
query_model_year           0
odiNumber                  0
manufacturer               0
products                   0
summary                    0
components                 0
dateComplaintFiled         0
dateOfIncident             0
numberOfDeaths             0
numberOfInjuries           0
fire                       0
crash                      0
query_make                 0
dtype: int64

In [44]:
def get_vehicle_year(products):
    for p in products:
        if p.get("type") == "Vehicle":
            return p.get("productYear")
    return None

df_clean["product_year"] = df_clean["products"].apply(
    get_vehicle_year
)

df_clean["product_year"] = pd.to_numeric(
    df_clean["product_year"], errors="coerce"
)

In [45]:
df_clean["year_match"] = (
    df_clean["model_year"] == df_clean["product_year"]
)

print(df_clean["year_match"].value_counts())

display(
    df_clean.loc[
        ~df_clean["year_match"],
        ["odiNumber", "make", "model",
         "model_year", "product_year"]
    ].head(10)
)

year_match
True    9461
Name: count, dtype: int64


,odiNumber,make,model,model_year,product_year


In [46]:
display(
    df_clean.groupby(["make", "model", "model_year"])
    .size()
    .reset_index(name="complaint_count")
)

,make,model,model_year,complaint_count
0,AUDI,Q5,2018,207
1,AUDI,Q5,2019,52
2,AUDI,Q5,2020,25
3,BMW,3 SERIES,2018,59
4,BMW,3 SERIES,2019,41
5,BMW,3 SERIES,2020,53
6,FORD,ESCAPE,2018,1811
7,FORD,ESCAPE,2019,595
8,FORD,ESCAPE,2020,1537
9,FORD,F-150,2018,84


In [47]:
# multiple vehicle products validation 
def count_vehicles(products):
    return sum(
        p.get("type") == "Vehicle"
        for p in products
        if isinstance(p, dict)
    )

df_clean["vehicle_count"] = (
    df_clean["products"].apply(count_vehicles)
)

print(df_clean["vehicle_count"].value_counts())

vehicle_count
1    9448
2      13
Name: count, dtype: int64


In [48]:
print("No vehicle:",  (df_clean["vehicle_count"] == 0).sum())

print("Multiple vehicles:", (df_clean["vehicle_count"] > 1).sum())

display(
    df_clean.loc[
        df_clean["vehicle_count"] > 1,
        ["odiNumber", "make", "model",
         "model_year", "vehicle_count"]
    ].head(10)
)

No vehicle: 0
Multiple vehicles: 13


,odiNumber,make,model,model_year,vehicle_count
278,11484105,TOYOTA,CAMRY,2018,2
872,11610002,TOYOTA,CAMRY,2019,2
1229,11647348,TOYOTA,CAMRY,2020,2
1243,11636054,TOYOTA,CAMRY,2020,2
2366,11763441,HONDA,CIVIC,2020,2
2493,11553620,HONDA,CIVIC,2020,2
3451,11621302,FORD,ESCAPE,2018,2
4427,11755101,FORD,ESCAPE,2019,2
7386,11556364,HYUNDAI,ELANTRA,2020,2
7859,11515878,NISSAN,ALTIMA,2019,2


In [49]:
multiple = df_clean[df_clean["vehicle_count"] > 1]

for _, row in multiple.iterrows():
    print("\nComplaint ID:", row["odiNumber"])
    print("Query:", row["query_make"],
          row["query_model"],
          row["query_model_year"])

    for p in row["products"]:
        if p.get("type") == "Vehicle":
            print("Vehicle:",
                  p.get("productMake"),
                  p.get("productModel"),
                  p.get("productYear"))


Complaint ID: 11484105
Query: Toyota Camry 2018
Vehicle: TOYOTA CAMRY 2018
Vehicle: TBD TBD 9999

Complaint ID: 11610002
Query: Toyota Camry 2019
Vehicle: TOYOTA CAMRY 2019
Vehicle: TBD TBD 9999

Complaint ID: 11647348
Query: Toyota Camry 2020
Vehicle: TOYOTA CAMRY 2020
Vehicle: TBD TBD 9999

Complaint ID: 11636054
Query: Toyota Camry 2020
Vehicle: TOYOTA CAMRY 2020
Vehicle: TBD TBD 9999

Complaint ID: 11763441
Query: Honda Civic 2020
Vehicle: HONDA CIVIC 2020
Vehicle: TBD_TEST TBD_Test 9999

Complaint ID: 11553620
Query: Honda Civic 2020
Vehicle: HONDA CIVIC 2020
Vehicle: TBD TBD 9999

Complaint ID: 11621302
Query: Ford Escape 2018
Vehicle: FORD ESCAPE 2018
Vehicle: TBD TBD 9999

Complaint ID: 11755101
Query: Ford Escape 2019
Vehicle: FORD ESCAPE 2019
Vehicle: TBD_TEST TBD_Test 9999

Complaint ID: 11556364
Query: Hyundai Elantra 2020
Vehicle: HYUNDAI ELANTRA 2020
Vehicle: TBD TBD 9999

Complaint ID: 11515878
Query: Nissan Altima 2019
Vehicle: NISSAN ALTIMA 2019
Vehicle: TBD TBD 9999


In [50]:
# Check placeholder vehicle entries
multiple = df_clean[df_clean["vehicle_count"] > 1]

for _, row in multiple.iterrows():
    vehicles = [
        p for p in row["products"]
        if isinstance(p, dict) and p.get("type") == "Vehicle"
    ]

    first = vehicles[0]

    assert first["productMake"].upper() == row["query_make"].upper()
    assert str(first["productYear"]) == str(row["query_model_year"])

print("All 13 records have matching first vehicle make and year.")

All 13 records have matching first vehicle make and year.


In [51]:
print("Model mismatches:",
      (multiple["model"] !=
       multiple["query_model"].str.upper()).sum())

Model mismatches: 0


### Multiple Vehicle Products Validation

Among the 9,461 complaints, 13 records contain two Vehicle products.

Manual inspection confirmed that the first Vehicle product matches the queried vehicle. The second Vehicle product contains placeholder values such as TBD and model year 9999.

Therefore, these records are retained without duplication, and the existing vehicle model extraction method is preserved.

(maybe add a visualization here)

## Component Cleaning

In [52]:
df_clean["components"] = (
    df_clean["components"]
    .str.upper()
    .str.strip()
)

display(df_clean["components"].value_counts().head(20))

components
ENGINE                                1719
POWER TRAIN                            820
UNKNOWN OR OTHER                       761
STRUCTURE                              641
STEERING                               424
ENGINE AND ENGINE COOLING              372
ELECTRICAL SYSTEM                      351
FUEL SYSTEM, GASOLINE                  297
FUEL/PROPULSION SYSTEM                 270
SERVICE BRAKES                         208
AIR BAGS                               196
POWER TRAIN,ENGINE                     186
EXTERIOR LIGHTING                      135
FORWARD COLLISION AVOIDANCE            110
VISIBILITY/WIPER                       105
ENGINE,FUEL/PROPULSION SYSTEM           95
UNKNOWN OR OTHER,ENGINE                 91
ELECTRICAL SYSTEM,UNKNOWN OR OTHER      86
WHEELS                                  79
ELECTRICAL SYSTEM,ENGINE                74
Name: count, dtype: int64

In [53]:
print("Missing:", df_clean["components"].isna().sum())
print("Blank:", (df_clean["components"] == "").sum())

print("UNKNOWN OR OTHER:",
      (df_clean["components"] == "UNKNOWN OR OTHER").sum())

Missing: 0
Blank: 0
UNKNOWN OR OTHER: 761


In [54]:
print("Total rows:", len(df_clean))
print("Unique IDs:", df_clean["odiNumber"].nunique())
print("Duplicate IDs:", df_clean["odiNumber"].duplicated().sum())

print("Missing filing dates:", df_clean["filed_date"].isna().sum())

print("Missing vehicle keys:")
print(df_clean[["make", "model", "model_year"]].isna().sum())

print("Invalid incident dates:",
      df_clean["invalid_incident_date"].sum())

Total rows: 9461
Unique IDs: 9461
Duplicate IDs: 0
Missing filing dates: 0
Missing vehicle keys:
make          0
model         0
model_year    0
dtype: int64
Invalid incident dates: 27


In [55]:
expected_model = df_clean["query_model"].str.upper().str.strip()

print("Model mismatches:",
      (df_clean["model"] != expected_model).sum())

display(
    df_clean.loc[
        df_clean["model"] != expected_model,
        ["odiNumber", "query_model", "product_model", "model"]
    ].head(10)
)

assert (df_clean["model"] == expected_model).all()

Model mismatches: 0


,odiNumber,query_model,product_model,model


## Export Cleaned Dataset

In [56]:
complaints_clean = df_clean.rename(columns={
    "odiNumber": "complaint_id",
    "numberOfInjuries": "number_of_injuries",
    "numberOfDeaths": "number_of_deaths"
}).copy()

# Use validated reporting delay for EDA
complaints_clean["reporting_delay"] = (
    complaints_clean["reporting_delay_clean"]
)

columns = [
    "complaint_id",
    "make",
    "model",
    "model_year",
    "product_model",
    "filed_date",
    "incident_date",
    "components",
    "crash",
    "fire",
    "number_of_injuries",
    "number_of_deaths",
    "summary",
    "reporting_delay",
    "invalid_incident_date"
]

complaints_clean = complaints_clean[columns]

display(complaints_clean.head())
print("Shape:", complaints_clean.shape)

,complaint_id,make,model,model_year,product_model,filed_date,incident_date,components,crash,fire,number_of_injuries,number_of_deaths,summary,reporting_delay,invalid_incident_date
0,11770133,TOYOTA,CAMRY,2018,CAMRY,2026-10-09,2026-09-29,POWER TRAIN,False,False,0,0,"Total transmission failure at about 107,770 mi...",10.0,False
1,11769969,TOYOTA,CAMRY,2018,CAMRY,2026-10-08,2026-10-06,ENGINE,False,False,0,0,My 2018 Toyota Camry LE went into Limp mode wh...,2.0,False
2,11769124,TOYOTA,CAMRY,2018,CAMRY,2026-10-05,2026-10-03,POWER TRAIN,False,False,0,0,The contact owns a 2018 Toyota Camry. The cont...,2.0,False
3,11768868,TOYOTA,CAMRY,2018,CAMRY,2026-10-04,2026-10-02,FUEL/PROPULSION SYSTEM,False,False,0,0,I am filing a formal complaint regarding a sev...,2.0,False
4,11768335,TOYOTA,CAMRY,2018,CAMRY,2026-10-01,2026-09-29,POWER TRAIN,False,False,0,0,"2018 Toyota Camry approx 126,000 miles. Trans...",2.0,False


Shape: (9461, 15)


In [57]:
print("Rows:", len(complaints_clean))
print("Duplicates:", complaints_clean["complaint_id"].duplicated().sum())
print("Missing filed dates:", complaints_clean["filed_date"].isna().sum())
print("Invalid incident dates:", complaints_clean["invalid_incident_date"].sum())

assert len(complaints_clean) == len(df_raw)
assert complaints_clean["complaint_id"].is_unique
assert complaints_clean["filed_date"].notna().all()
assert df_clean["year_match"].all()

print("Validation passed!")

Rows: 9461
Duplicates: 0
Missing filed dates: 0
Invalid incident dates: 27
Validation passed!


In [58]:
output_file = PROCESSED_DIR / "complaints_clean.csv"

complaints_clean.to_csv(output_file, index=False)

print("Saved:", output_file)

Saved: /Users/xinyishen/Desktop/Brown/DATA 1030/DATA1030-fall2026/DATA1030-Midterm-Xinfinity/data/processed/complaints_clean.csv


In [59]:
df_check = pd.read_csv(
    output_file,
    parse_dates=["filed_date", "incident_date"]
)

display(df_check.head())
print(df_check.shape)

,complaint_id,make,model,model_year,product_model,filed_date,incident_date,components,crash,fire,number_of_injuries,number_of_deaths,summary,reporting_delay,invalid_incident_date
0,11770133,TOYOTA,CAMRY,2018,CAMRY,2026-10-09,2026-09-29,POWER TRAIN,False,False,0,0,"Total transmission failure at about 107,770 mi...",10.0,False
1,11769969,TOYOTA,CAMRY,2018,CAMRY,2026-10-08,2026-10-06,ENGINE,False,False,0,0,My 2018 Toyota Camry LE went into Limp mode wh...,2.0,False
2,11769124,TOYOTA,CAMRY,2018,CAMRY,2026-10-05,2026-10-03,POWER TRAIN,False,False,0,0,The contact owns a 2018 Toyota Camry. The cont...,2.0,False
3,11768868,TOYOTA,CAMRY,2018,CAMRY,2026-10-04,2026-10-02,FUEL/PROPULSION SYSTEM,False,False,0,0,I am filing a formal complaint regarding a sev...,2.0,False
4,11768335,TOYOTA,CAMRY,2018,CAMRY,2026-10-01,2026-09-29,POWER TRAIN,False,False,0,0,"2018 Toyota Camry approx 126,000 miles. Trans...",2.0,False


(9461, 15)


In [60]:
print("DATA COLLECTION SUMMARY")
print("---")
print("API:", BASE_URL)
print("Retrieval date (UTC):", retrieval_date)
print("Total complaints:", len(df_clean))
print("Vehicle groups:", df_clean.groupby(
    ["make", "model", "model_year"]
).ngroups)

print("Earliest filing date:", earliest_date.date())
print("Latest filing date:", latest_date.date())

DATA COLLECTION SUMMARY
---
API: https://api.nhtsa.gov/complaints/complaintsByVehicle
Retrieval date (UTC): 2026-10-14 21:49:03.049725+00:00
Total complaints: 9461
Vehicle groups: 30
Earliest filing date: 2017-08-04
Latest filing date: 2026-10-12
